# 1. Title and Project Overview

This notebook builds a production-style Big Data financial pipeline for Argentine markets using Python, BigQuery, and dbt Core.

## Project purpose
- Ingest market and macro data (Argentina + global factors)
- Load raw data into BigQuery
- Transform data into processed and analytics layers
- Compute stock and portfolio risk-return metrics

## Architecture summary
- raw layer: immutable ingested tables
- processed layer: cleaned returns and risk features
- analytics/serving layer: portfolio scenarios and rankings

## Data sources
- Yahoo Finance via yfinance: MERVAL proxy, ARS/USD, VIX, EEM, US 10Y, selected stocks

## Expected outputs
- BigQuery raw/processed/analytics tables
- dbt models built on top of raw data
- metrics ready for a future application

# 2. Imports and Configuration

In [1]:
import os
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
from google.api_core.exceptions import NotFound
from google.cloud import bigquery
from google.oauth2 import service_account

# Date range controls the entire ingestion window.
START_DATE = "2018-01-01"
END_DATE = "2026-01-01"

# Ingestion mode: False = WRITE_TRUNCATE (full reload), True = WRITE_APPEND (incremental)
# Set to True for production pipelines, False for notebook/demo runs
INCREMENTAL_MODE = False

# BigQuery placeholders. Replace with your own project and region.
PROJECT_ID = "bigdata-financeargentina"
BQ_LOCATION = "US"

DATASETS = {
    "raw": "raw_market",
    "processed": "processed_market",
    "analytics": "analytics_market"
}

RAW_TABLES = {
    "merval_prices": "merval_prices",
    "fx_prices": "fx_prices",
    "vix_prices": "vix_prices",
    "eem_prices": "eem_prices",
    "rf_rates": "rf_rates",
    "stock_prices": "stock_prices",
    "factor_prices": "factor_prices"
}

PROCESSED_TABLES = {
    "asset_returns": "asset_returns",
    "factor_returns": "factor_returns",
    "stock_metrics": "stock_metrics",
    "beta_metrics": "beta_metrics",
    "correlation_matrix_long": "correlation_matrix_long"
}

ANALYTICS_TABLES = {
    "portfolio_inputs": "portfolio_inputs",
    "portfolio_scenarios": "portfolio_scenarios",
    "stock_rankings": "stock_rankings"
}

# User-configurable stock universe.
selected_stocks = ["GGAL.BA", "YPFD.BA", "PAMP.BA", "BMA.BA", "CEPU.BA"]

# Raw symbols for market and factor ingestion.
SYMBOLS = {
    "MERVAL": "^MERV",
    "USDARS": "ARS=X",
    "VIX": "^VIX",
    "EEM": "EEM",
    "US10Y": "^TNX"
}

ingestion_timestamp = datetime.now(timezone.utc)
run_id = f"run_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

print(f"Run ID: {run_id}")
print(f"Ingestion timestamp (UTC): {ingestion_timestamp.isoformat()}")

Run ID: run_20260421_141440_1b65423a
Ingestion timestamp (UTC): 2026-04-21T14:14:40.010702+00:00


# 3. BigQuery Connection Setup

Authentication options supported in this notebook:
- Application Default Credentials (ADC)
- Service account key using environment variable GOOGLE_APPLICATION_CREDENTIALS

No secrets are hardcoded in this notebook.

In [2]:
def build_bigquery_client(project_id: str) -> bigquery.Client:
    """Create a BigQuery client using service account key if provided, otherwise ADC."""
    sa_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS", "").strip()

    if sa_path:
        # Option 1: explicit service-account key file (recommended for controlled deployments).
        credentials = service_account.Credentials.from_service_account_file(sa_path)
        return bigquery.Client(project=project_id, credentials=credentials)

    # Option 2: ADC (recommended for local development and cloud runtimes with IAM bindings).
    return bigquery.Client(project=project_id)


def ensure_dataset(client: bigquery.Client, project_id: str, dataset_id: str, location: str) -> None:
    dataset_ref = bigquery.Dataset(f"{project_id}.{dataset_id}")
    dataset_ref.location = location
    client.create_dataset(dataset_ref, exists_ok=True)


client = build_bigquery_client(PROJECT_ID)
for ds in DATASETS.values():
    ensure_dataset(client, PROJECT_ID, ds, BQ_LOCATION)

print("BigQuery client ready and datasets ensured.")

BigQuery client ready and datasets ensured.


# 4. Raw Data Ingestion

This section downloads and standardizes raw data from Yahoo Finance.

Standard schema for asset price tables:
- date, ticker, close, adj_close, volume, source, ingestion_timestamp, run_id, currency

Standard schema for factor tables:
- date, factor_name, value, source, ingestion_timestamp, run_id

In [3]:
def safe_download(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Download from yfinance and return a flat DataFrame, or empty DataFrame if unavailable."""
    df = yf.download(symbol, start=start_date, end=end_date, auto_adjust=False, progress=False)
    if df is None or df.empty:
        return pd.DataFrame()

    df = df.reset_index()

    # yfinance can return MultiIndex columns; flatten only when needed.
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            c[0] if c[1] == "" else f"{c[0]}_{c[1]}"
            for c in df.columns
        ]

    return df


def standardize_asset_prices(
    raw_df: pd.DataFrame,
    ticker: str,
    currency: str,
    source: str = "yfinance"
) -> pd.DataFrame:
    """Map raw yfinance data to the project standard asset price schema."""
    required_date_col = "Date" if "Date" in raw_df.columns else "date"

    close_candidates = [c for c in raw_df.columns if str(c).startswith("Close") or c == "Close"]
    adj_close_candidates = [c for c in raw_df.columns if str(c).startswith("Adj Close") or c == "Adj Close"]
    volume_candidates = [c for c in raw_df.columns if str(c).startswith("Volume") or c == "Volume"]

    if not close_candidates:
        raise ValueError(f"Missing close column for ticker {ticker}. Available columns: {list(raw_df.columns)}")

    standardized = pd.DataFrame({
        "date": pd.to_datetime(raw_df[required_date_col], errors="coerce"),
        "ticker": ticker,
        "close": pd.to_numeric(raw_df[close_candidates[0]], errors="coerce"),
        "adj_close": pd.to_numeric(raw_df[adj_close_candidates[0]], errors="coerce") if adj_close_candidates else np.nan,
        "volume": pd.to_numeric(raw_df[volume_candidates[0]], errors="coerce") if volume_candidates else np.nan,
        "source": source,
        "ingestion_timestamp": ingestion_timestamp,
        "run_id": run_id,
        "currency": currency
    })

    standardized = standardized.dropna(subset=["date", "close"]).sort_values("date")
    return standardized


def factor_from_asset(asset_df: pd.DataFrame, factor_name: str) -> pd.DataFrame:
    """Convert standardized asset prices into factor-name/value long format."""
    factor_df = asset_df[["date", "close"]].copy()
    factor_df.rename(columns={"close": "value"}, inplace=True)
    factor_df["factor_name"] = factor_name
    factor_df["source"] = "yfinance"
    factor_df["ingestion_timestamp"] = ingestion_timestamp
    factor_df["run_id"] = run_id
    return factor_df[["date", "factor_name", "value", "source", "ingestion_timestamp", "run_id"]]


def assert_not_empty(df: pd.DataFrame, name: str) -> None:
    if df.empty:
        raise ValueError(f"{name} is empty. Check ticker availability or date range.")

In [4]:
# Download and standardize the main market and macro factors.
merval_raw = safe_download(SYMBOLS["MERVAL"], START_DATE, END_DATE)
fx_raw = safe_download(SYMBOLS["USDARS"], START_DATE, END_DATE)
vix_raw = safe_download(SYMBOLS["VIX"], START_DATE, END_DATE)
eem_raw = safe_download(SYMBOLS["EEM"], START_DATE, END_DATE)
rf_raw = safe_download(SYMBOLS["US10Y"], START_DATE, END_DATE)

merval_prices = standardize_asset_prices(merval_raw, "MERVAL", "ARS")
fx_prices = standardize_asset_prices(fx_raw, "USDARS", "ARS_PER_USD")
vix_prices = standardize_asset_prices(vix_raw, "VIX", "INDEX")
eem_prices = standardize_asset_prices(eem_raw, "EEM", "USD")
rf_rates = standardize_asset_prices(rf_raw, "US10Y", "PCT")

for df_name, obj in [
    ("merval_prices", merval_prices),
    ("fx_prices", fx_prices),
    ("vix_prices", vix_prices),
    ("eem_prices", eem_prices),
    ("rf_rates", rf_rates)
]:
    assert_not_empty(obj, df_name)

# Download and standardize user-selected stocks.
stock_frames = []
for ticker in selected_stocks:
    stock_raw = safe_download(ticker, START_DATE, END_DATE)
    if stock_raw.empty:
        print(f"Warning: no data for {ticker}. It will be skipped.")
        continue
    stock_frames.append(standardize_asset_prices(stock_raw, ticker, "ARS"))

stock_prices = pd.concat(stock_frames, ignore_index=True) if stock_frames else pd.DataFrame()
assert_not_empty(stock_prices, "stock_prices")

# Factor long table used later by dbt staging model.
factor_prices = pd.concat([
    factor_from_asset(merval_prices, "MERVAL"),
    factor_from_asset(fx_prices, "USDARS"),
    factor_from_asset(vix_prices, "VIX"),
    factor_from_asset(eem_prices, "EEM"),
    factor_from_asset(rf_rates, "US10Y")
], ignore_index=True)

# Optional placeholder for future country-risk feed integration.
factor_prices["country_risk_proxy"] = np.where(factor_prices["factor_name"] == "US10Y", np.nan, np.nan)

print("Raw ingestion complete.")
display(stock_prices.head())
display(factor_prices.head())

Raw ingestion complete.


,date,ticker,close,adj_close,volume,source,ingestion_timestamp,run_id,currency
0,2018-01-01,GGAL.BA,124.400002,91.743637,0,yfinance,2026-04-21 14:14:40.010702+00:00,run_20260421_141440_1b65423a,ARS
1,2018-01-02,GGAL.BA,124.449997,91.780540,393713,yfinance,2026-04-21 14:14:40.010702+00:00,run_20260421_141440_1b65423a,ARS
2,2018-01-03,GGAL.BA,123.550003,91.116791,453943,yfinance,2026-04-21 14:14:40.010702+00:00,run_20260421_141440_1b65423a,ARS
3,2018-01-04,GGAL.BA,129.899994,95.799835,636362,yfinance,2026-04-21 14:14:40.010702+00:00,run_20260421_141440_1b65423a,ARS
4,2018-01-05,GGAL.BA,129.750000,95.689240,657700,yfinance,2026-04-21 14:14:40.010702+00:00,run_20260421_141440_1b65423a,ARS


,date,factor_name,value,source,ingestion_timestamp,run_id,country_risk_proxy
0,2018-01-02,MERVAL,31084.0,yfinance,2026-04-21 14:14:40.010702+00:00,run_20260421_141440_1b65423a,NaN
1,2018-01-03,MERVAL,31476.0,yfinance,2026-04-21 14:14:40.010702+00:00,run_20260421_141440_1b65423a,NaN
2,2018-01-04,MERVAL,31952.0,yfinance,2026-04-21 14:14:40.010702+00:00,run_20260421_141440_1b65423a,NaN
3,2018-01-05,MERVAL,32190.0,yfinance,2026-04-21 14:14:40.010702+00:00,run_20260421_141440_1b65423a,NaN
4,2018-01-08,MERVAL,32351.0,yfinance,2026-04-21 14:14:40.010702+00:00,run_20260421_141440_1b65423a,NaN


# 5. Raw Uploads to BigQuery

Raw tables are loaded with `WRITE_TRUNCATE` for reproducible notebook runs in academic workflows.
For production incremental pipelines, switch selected tables to `WRITE_APPEND` and partition by date.

In [5]:
def to_bq(
    client: bigquery.Client,
    dataframe: pd.DataFrame,
    project_id: str,
    dataset: str,
    table: str,
    write_disposition: str = "WRITE_TRUNCATE"
) -> None:
    if dataframe.empty:
        raise ValueError(f"Cannot upload empty DataFrame to {dataset}.{table}")

    table_id = f"{project_id}.{dataset}.{table}"
    job_config = bigquery.LoadJobConfig(write_disposition=write_disposition)

    try:
        existing_table = client.get_table(table_id)
    except NotFound:
        existing_table = None

    if write_disposition == "WRITE_APPEND" and existing_table is not None:
        existing_columns = {field.name for field in existing_table.schema}
        new_columns = [column for column in dataframe.columns if column not in existing_columns]

        if new_columns:
            job_config.schema_update_options = [bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION]
            print(
                f"Schema evolution enabled for {table_id}. Adding columns: {', '.join(new_columns)}"
            )

    job = client.load_table_from_dataframe(dataframe, table_id, job_config=job_config)
    job.result()
    print(f"Uploaded {len(dataframe):,} rows -> {table_id} ({write_disposition})")


# Determine write disposition based on INCREMENTAL_MODE
# False (default): WRITE_TRUNCATE for full reload in academic/demo mode
# True: WRITE_APPEND for production incremental updates
raw_write_mode = "WRITE_APPEND" if INCREMENTAL_MODE else "WRITE_TRUNCATE"

to_bq(client, merval_prices, PROJECT_ID, DATASETS["raw"], RAW_TABLES["merval_prices"], raw_write_mode)
to_bq(client, fx_prices, PROJECT_ID, DATASETS["raw"], RAW_TABLES["fx_prices"], raw_write_mode)
to_bq(client, vix_prices, PROJECT_ID, DATASETS["raw"], RAW_TABLES["vix_prices"], raw_write_mode)
to_bq(client, eem_prices, PROJECT_ID, DATASETS["raw"], RAW_TABLES["eem_prices"], raw_write_mode)
to_bq(client, rf_rates, PROJECT_ID, DATASETS["raw"], RAW_TABLES["rf_rates"], raw_write_mode)
to_bq(client, stock_prices, PROJECT_ID, DATASETS["raw"], RAW_TABLES["stock_prices"], raw_write_mode)
to_bq(client, factor_prices, PROJECT_ID, DATASETS["raw"], RAW_TABLES["factor_prices"], raw_write_mode)

print("Raw layer upload finished.")

Uploaded 1,946 rows -> bigdata-financeargentina.raw_market.merval_prices (WRITE_TRUNCATE)
Uploaded 2,083 rows -> bigdata-financeargentina.raw_market.fx_prices (WRITE_TRUNCATE)
Uploaded 2,011 rows -> bigdata-financeargentina.raw_market.vix_prices (WRITE_TRUNCATE)
Uploaded 2,011 rows -> bigdata-financeargentina.raw_market.eem_prices (WRITE_TRUNCATE)
Uploaded 2,011 rows -> bigdata-financeargentina.raw_market.rf_rates (WRITE_TRUNCATE)
Uploaded 9,750 rows -> bigdata-financeargentina.raw_market.stock_prices (WRITE_TRUNCATE)
Uploaded 10,062 rows -> bigdata-financeargentina.raw_market.factor_prices (WRITE_TRUNCATE)
Raw layer upload finished.


# 6. Processed Feature Engineering in Python

Key formulas:
- log return: ln(P_t / P_{t-1})
- usd_adjusted_return = local_stock_return - usdars_return
- excess_return = stock_return - risk_free_rate

In [6]:
def compute_log_returns(df: pd.DataFrame, price_col: str, group_col: str = None) -> pd.Series:
    def _compute(series: pd.Series) -> pd.Series:
        prev = series.shift(1)
        valid = series.gt(0) & prev.gt(0)
        out = pd.Series(np.nan, index=series.index, dtype="float64")
        out.loc[valid] = np.log(series.loc[valid] / prev.loc[valid])
        return out

    if group_col:
        return df.groupby(group_col)[price_col].transform(_compute)
    return _compute(df[price_col])


# Asset returns for user-selected stocks.
asset_returns = stock_prices[["date", "ticker", "close"]].copy()
asset_returns["date"] = pd.to_datetime(asset_returns["date"])
asset_returns = asset_returns.sort_values(["ticker", "date"])
asset_returns["log_return"] = compute_log_returns(asset_returns, "close", "ticker")

# Factor returns.
factor_base = factor_prices[["date", "factor_name", "value"]].copy()
factor_base["date"] = pd.to_datetime(factor_base["date"])
factor_base = factor_base.sort_values(["factor_name", "date"])
factor_base["factor_return"] = compute_log_returns(factor_base, "value", "factor_name")

# Pivot factors for easier metric calculations.
factor_wide = factor_base.pivot(index="date", columns="factor_name", values="factor_return").reset_index()

# Merge stock and factor returns.
returns_enriched = asset_returns.merge(factor_wide, on="date", how="left")

# USD-adjusted return for Argentine stocks.
returns_enriched["usd_adjusted_return"] = returns_enriched["log_return"] - returns_enriched.get("USDARS", 0)

# Convert US10Y series to an approximate daily risk-free rate.
rf_level = factor_prices[factor_prices["factor_name"] == "US10Y"][["date", "value"]].copy()
rf_level["date"] = pd.to_datetime(rf_level["date"])
rf_level = rf_level.sort_values("date")

# Yahoo ^TNX may be reported as 10x percent (for example 43 = 4.3%).
rf_level["rf_annual"] = rf_level["value"]
if rf_level["rf_annual"].median(skipna=True) > 1:
    rf_level["rf_annual"] = rf_level["rf_annual"] / 100
if rf_level["rf_annual"].median(skipna=True) > 0.2:
    rf_level["rf_annual"] = rf_level["rf_annual"] / 10

rf_level["risk_free_daily"] = rf_level["rf_annual"] / 252

returns_enriched = returns_enriched.merge(rf_level[["date", "risk_free_daily"]], on="date", how="left")
returns_enriched["risk_free_daily"] = returns_enriched["risk_free_daily"].ffill()
returns_enriched["excess_return"] = returns_enriched["log_return"] - returns_enriched["risk_free_daily"]

# USD-adjusted MERVAL return for context.
returns_enriched["merval_usd_adjusted_return"] = returns_enriched.get("MERVAL", np.nan) - returns_enriched.get("USDARS", np.nan)

# Optional placeholders for future macro feeds.
returns_enriched["inflation_proxy"] = returns_enriched.get("USDARS", np.nan)
returns_enriched["country_risk_proxy"] = returns_enriched.get("VIX", np.nan)

processed_asset_returns = returns_enriched[[
    "date", "ticker", "close", "log_return", "usd_adjusted_return",
    "excess_return", "MERVAL", "EEM", "VIX", "USDARS", "risk_free_daily",
    "merval_usd_adjusted_return", "inflation_proxy", "country_risk_proxy"
]].copy()

# The first observation for each ticker or factor has no previous price, so its return is null by design.
# Filter those rows out before writing the processed layer so downstream queries do not see null returns.
processed_asset_returns = processed_asset_returns.dropna(subset=["log_return"]).reset_index(drop=True)

processed_factor_returns = factor_base[["date", "factor_name", "value", "factor_return"]].copy()
processed_factor_returns = processed_factor_returns.dropna(subset=["factor_return"]).reset_index(drop=True)

display(processed_asset_returns.head())
display(processed_factor_returns.head())

,date,ticker,close,log_return,usd_adjusted_return,excess_return,MERVAL,EEM,VIX,USDARS,risk_free_daily,merval_usd_adjusted_return,inflation_proxy,country_risk_proxy
0,2018-01-02,BMA.BA,216.550003,-0.011250,0.015080,-0.011348,NaN,NaN,NaN,-0.026330,0.000098,NaN,-0.026330,NaN
1,2018-01-03,BMA.BA,216.899994,0.001615,0.012458,0.001518,0.012532,0.009536,-0.065563,-0.010843,0.000097,0.023376,-0.010843,-0.065563
2,2018-01-04,BMA.BA,218.550003,0.007578,0.004701,0.007481,0.015009,0.004939,0.007621,0.002878,0.000097,0.012132,0.002878,0.007621
3,2018-01-05,BMA.BA,219.750000,0.005476,-0.003726,0.005377,0.007421,0.008586,0.000000,0.009202,0.000098,-0.001781,0.009202,0.000000
4,2018-01-08,BMA.BA,217.850006,-0.008684,-0.006694,-0.008782,0.004989,0.000000,0.032020,-0.001990,0.000098,0.006979,-0.001990,0.032020


,date,factor_name,value,factor_return
0,2018-01-03,EEM,48.470001,0.009536
1,2018-01-04,EEM,48.709999,0.004939
2,2018-01-05,EEM,49.130001,0.008586
3,2018-01-08,EEM,49.130001,0.000000
4,2018-01-09,EEM,49.049999,-0.001630


# 7. Stock-Level Metrics

For each selected stock we compute:
- average return
- volatility
- beta vs MERVAL
- beta vs EEM
- correlation with MERVAL
- correlation with FX
- Sharpe ratio
- max drawdown

In [7]:
def regression_beta(y: pd.Series, x: pd.Series) -> float:
    reg_df = pd.concat([y, x], axis=1).dropna()
    if reg_df.shape[0] < 20:
        return np.nan
    y_clean = reg_df.iloc[:, 0]
    x_clean = sm.add_constant(reg_df.iloc[:, 1])
    model = sm.OLS(y_clean, x_clean).fit()
    return float(model.params.iloc[1])


def max_drawdown_from_returns(return_series: pd.Series) -> float:
    """Calculate maximum drawdown from a return series.
    
    Max drawdown measures the largest peak-to-trough decline.
    Used in Calmar Ratio calculation for risk-adjusted returns.
    """
    clean = return_series.dropna()
    if clean.empty:
        return np.nan
    cumulative = np.exp(clean.cumsum())
    running_max = cumulative.cummax()
    drawdown = cumulative / running_max - 1.0
    return float(drawdown.min())


def downside_stddev(return_series: pd.Series, target_return: float = 0) -> float:
    """Calculate downside deviation (only considers returns below target).
    
    Sortino Ratio uses downside deviation instead of total volatility.
    This is more appropriate for skewed return distributions as it only
    penalizes negative returns, not upside volatility.
    
    Args:
        return_series: Daily returns
        target_return: Threshold return (default 0 = risk-free)
    
    Returns:
        Annualized downside deviation
    """
    clean = return_series.dropna()
    if clean.empty:
        return np.nan
    # Only consider negative returns (returns below target)
    downside_returns = clean[clean < target_return]
    if len(downside_returns) == 0:
        return 0.0
    downside_deviation = downside_returns.std()
    return float(downside_deviation * np.sqrt(252))


def downside_deviation(return_series: pd.Series, target_return: float = 0) -> float:
    """
    Calculate downside deviation (only negative deviations from target).
    Used for Sortino Ratio calculation.
    
    Args:
        return_series: Daily return series
        target_return: Threshold return (default 0)
    
    Returns:
        Downside standard deviation annualized
    """
    clean = return_series.dropna()
    if clean.empty:
        return np.nan
    negative_returns = clean[clean < target_return]
    if len(negative_returns) < 2:
        return np.nan
    downside_std = negative_returns.std()
    return float(downside_std * np.sqrt(252))


# Get risk-free rate for Sortino calculation
rf_daily_mean = processed_asset_returns["risk_free_daily"].mean(skipna=True)


def calculate_sortino(returns: pd.Series, rf_daily: pd.Series) -> float:
    """
    Calculate Sortino Ratio: (Annualized Return - Risk-Free Rate) / Downside Deviation
    
    Sortino uses only downside volatility, making it better for asymmetric distributions
    where upside volatility should not penalize the risk-adjusted metric.
    
    Args:
        returns: Daily log returns
        rf_daily: Daily risk-free rate series
    
    Returns:
        Sortino Ratio (annualized)
    """
    clean = returns.dropna()
    if clean.empty:
        return np.nan
    
    rf_mean = rf_daily.mean(skipna=True) if not rf_daily.empty else 0
    annualized_excess_return = clean.mean() * 252 - rf_mean * 252
    
    dd = downside_deviation(clean)
    if dd == 0 or np.isnan(dd):
        return np.nan
    
    return float(annualized_excess_return / dd)


def calculate_calmar(returns: pd.Series) -> float:
    """
    Calculate Calmar Ratio: Annualized Return / |Maximum Drawdown|
    
    Measures return per unit of worst-case loss. Critical for emerging markets
    where tail risk is often underpriced.
    
    Args:
        returns: Daily log returns
    
    Returns:
        Calmar Ratio (annualized)
    """
    clean = returns.dropna()
    if clean.empty:
        return np.nan
    
    annualized_return = clean.mean() * 252
    mdd = max_drawdown_from_returns(clean)
    
    if mdd == 0 or np.isnan(mdd):
        return np.nan
    
    return float(annualized_return / abs(mdd))


rows = []
for ticker, grp in processed_asset_returns.groupby("ticker"):
    g = grp.sort_values("date").copy()

    avg_return_annual = g["log_return"].mean(skipna=True) * 252
    volatility_annual = g["log_return"].std(skipna=True) * np.sqrt(252)
    sharpe = np.nan
    if g["excess_return"].std(skipna=True) not in [0, np.nan]:
        sharpe = (g["excess_return"].mean(skipna=True) * 252) / (g["log_return"].std(skipna=True) * np.sqrt(252))

    rows.append({
        "ticker": ticker,
        "average_return": avg_return_annual,
        "volatility": volatility_annual,
        "beta_vs_merval": regression_beta(g["log_return"], g["MERVAL"]),
        "beta_vs_eem": regression_beta(g["log_return"], g["EEM"]),
        "corr_with_merval": g[["log_return", "MERVAL"]].corr().iloc[0, 1] if g[["log_return", "MERVAL"]].dropna().shape[0] > 3 else np.nan,
        "corr_with_fx": g[["log_return", "USDARS"]].corr().iloc[0, 1] if g[["log_return", "USDARS"]].dropna().shape[0] > 3 else np.nan,
        "sharpe_ratio": sharpe,
        "max_drawdown": max_drawdown_from_returns(g["log_return"]),
        "sortino_ratio": calculate_sortino(g["log_return"], processed_asset_returns["risk_free_daily"]),
        "calmar_ratio": calculate_calmar(g["log_return"]),
        "inflation_proxy": g["inflation_proxy"].mean(skipna=True) if "inflation_proxy" in g.columns else np.nan,
        "country_risk_proxy": g["country_risk_proxy"].mean(skipna=True) if "country_risk_proxy" in g.columns else np.nan
    })

stock_metrics = pd.DataFrame(rows).sort_values("ticker").reset_index(drop=True)
beta_metrics = stock_metrics[["ticker", "beta_vs_merval", "beta_vs_eem"]].copy()

display(stock_metrics)

,ticker,average_return,volatility,beta_vs_merval,beta_vs_eem,corr_with_merval,corr_with_fx,sharpe_ratio,max_drawdown,sortino_ratio,calmar_ratio,inflation_proxy,country_risk_proxy
0,BMA.BA,0.533550,0.659160,1.226500,0.962794,0.886416,0.038960,0.766193,-0.614650,0.998121,0.868055,0.002258,-0.000093
1,CEPU.BA,0.570019,0.626836,1.115562,0.806302,0.848996,0.039385,0.863881,-0.659236,1.100317,0.864666,0.002258,-0.000093
2,GGAL.BA,0.541546,0.627942,1.192401,0.958328,0.904586,0.007596,0.817018,-0.684045,1.029080,0.791683,0.002258,-0.000093
3,PAMP.BA,0.601602,0.563061,1.028409,0.686464,0.870155,0.042220,1.017821,-0.520472,1.293475,1.155877,0.002258,-0.000093
4,YPFD.BA,0.628713,0.570345,0.972664,0.890568,0.812950,0.015881,1.052357,-0.730358,1.520809,0.860829,0.002258,-0.000093


# 8. Correlation Matrix

Create pairwise stock correlation output in tidy/long format: ticker_1, ticker_2, correlation.

In [8]:
returns_wide = processed_asset_returns.pivot(
    index="date",
    columns="ticker",
    values="log_return"
)

corr_matrix = returns_wide.corr(min_periods=20)

correlation_matrix_long = (
    corr_matrix
    .rename_axis(index="ticker_1", columns="ticker_2")
    .stack()
    .reset_index(name="correlation")
)

display(correlation_matrix_long.head(20))

,ticker_1,ticker_2,correlation
0,BMA.BA,BMA.BA,1.000000
1,BMA.BA,CEPU.BA,0.715623
2,BMA.BA,GGAL.BA,0.901169
3,BMA.BA,PAMP.BA,0.749510
4,BMA.BA,YPFD.BA,0.678122
5,CEPU.BA,BMA.BA,0.715623
6,CEPU.BA,CEPU.BA,1.000000
7,CEPU.BA,GGAL.BA,0.742070
8,CEPU.BA,PAMP.BA,0.759241
9,CEPU.BA,YPFD.BA,0.655136


# 9. Portfolio Analytics

User can define custom portfolio weights and compare risk-return behavior.

In [9]:
user_portfolio = {
    "GGAL.BA": 0.25,
    "YPFD.BA": 0.25,
    "PAMP.BA": 0.20,
    "BMA.BA": 0.15,
    "CEPU.BA": 0.15
}

portfolio_id = f"portfolio_{run_id}"

weights = pd.Series(user_portfolio, name="weight").sort_index()

if not np.isclose(weights.sum(), 1.0):
    raise ValueError("Portfolio weights must sum to 1.0")

portfolio_inputs = weights.reset_index()
portfolio_inputs.columns = ["ticker", "weight"]
portfolio_inputs["portfolio_id"] = portfolio_id
portfolio_inputs["run_id"] = run_id
portfolio_inputs["ingestion_timestamp"] = ingestion_timestamp

missing_tickers = sorted(set(weights.index) - set(returns_wide.columns))
if missing_tickers:
    raise ValueError(f"These portfolio tickers are missing from returns_wide: {missing_tickers}")

portfolio_returns_wide = returns_wide.loc[:, weights.index].dropna(how="any")
if portfolio_returns_wide.empty:
    raise ValueError("No overlapping return history across all portfolio stocks.")

weights_vec = weights.values
mu_daily = portfolio_returns_wide.mean().values
cov_daily = portfolio_returns_wide.cov().values

portfolio_return_annual = float(np.dot(weights_vec, mu_daily) * 252)
portfolio_vol_annual = float(
    np.sqrt(np.dot(weights_vec.T, np.dot(cov_daily * 252, weights_vec)))
)

beta_lookup = stock_metrics[["ticker", "beta_vs_merval"]].drop_duplicates()
weighted_beta = float(
    portfolio_inputs.merge(beta_lookup, on="ticker", how="left")
    .assign(w_beta=lambda d: d["weight"] * d["beta_vs_merval"])
    ["w_beta"].sum()
)

portfolio_series = portfolio_returns_wide.dot(weights_vec)

rf_aligned = (
    processed_asset_returns[["date", "risk_free_daily"]]
    .dropna(subset=["risk_free_daily"])
    .groupby("date", as_index=True)["risk_free_daily"]
    .first()
    .reindex(portfolio_series.index)
    .ffill()
    .fillna(0)
)

aligned_portfolio = pd.concat(
    [portfolio_series.rename("portfolio_return"), rf_aligned.rename("risk_free_daily")],
    axis=1
).dropna()

aligned_portfolio["portfolio_excess"] = (
    aligned_portfolio["portfolio_return"] - aligned_portfolio["risk_free_daily"]
)

weighted_sharpe = float(
    (aligned_portfolio["portfolio_excess"].mean() * 252) /
    (aligned_portfolio["portfolio_return"].std() * np.sqrt(252))
)

portfolio_var_annual = float(
    np.dot(weights_vec.T, np.dot(cov_daily * 252, weights_vec))
)

marginal_contrib = np.dot(cov_daily * 252, weights_vec)
risk_contrib_variance = weights_vec * marginal_contrib
risk_contrib_pct = (
    risk_contrib_variance / portfolio_var_annual
    if portfolio_var_annual > 0
    else np.full(len(weights_vec), np.nan)
)

contribution_to_return = weights_vec * (mu_daily * 252)

individual_vol_annual = portfolio_returns_wide.std().values * np.sqrt(252)
weighted_avg_individual_vol = float(np.dot(weights_vec, individual_vol_annual))
diversification_effect = weighted_avg_individual_vol - portfolio_vol_annual
concentration_risk_hhi = float(np.sum(weights_vec ** 2))
num_assets = int(len(weights_vec))

portfolio_contributions = pd.DataFrame({
    "portfolio_id": portfolio_id,
    "ticker": weights.index,
    "weight": weights_vec,
    "contribution_to_return": contribution_to_return,
    "contribution_to_risk_pct": risk_contrib_pct,
    "run_id": run_id,
    "ingestion_timestamp": ingestion_timestamp
})

portfolio_scenarios = pd.DataFrame([{
    "portfolio_id": portfolio_id,
    "run_id": run_id,
    "expected_portfolio_return": portfolio_return_annual,
    "portfolio_volatility": portfolio_vol_annual,
    "weighted_beta": weighted_beta,
    "weighted_sharpe": weighted_sharpe,
    "diversification_effect": diversification_effect,
    "concentration_risk_hhi": concentration_risk_hhi,
    "num_assets": num_assets,
    "ingestion_timestamp": ingestion_timestamp
}])

display(portfolio_scenarios)
display(portfolio_contributions)

,portfolio_id,run_id,expected_portfolio_return,portfolio_volatility,weighted_beta,weighted_sharpe,diversification_effect,concentration_risk_hhi,num_assets,ingestion_timestamp
0,portfolio_run_20260421_141440_1b65423a,run_20260421_141440_1b65423a,0.578421,0.536886,1.098257,1.024266,0.068198,0.21,5,2026-04-21 14:14:40.010702+00:00


,portfolio_id,ticker,weight,contribution_to_return,contribution_to_risk_pct,run_id,ingestion_timestamp
0,portfolio_run_20260421_141440_1b65423a,BMA.BA,0.15,0.080033,0.167642,run_20260421_141440_1b65423a,2026-04-21 14:14:40.010702+00:00
1,portfolio_run_20260421_141440_1b65423a,CEPU.BA,0.15,0.085503,0.150113,run_20260421_141440_1b65423a,2026-04-21 14:14:40.010702+00:00
2,portfolio_run_20260421_141440_1b65423a,GGAL.BA,0.25,0.135387,0.271649,run_20260421_141440_1b65423a,2026-04-21 14:14:40.010702+00:00
3,portfolio_run_20260421_141440_1b65423a,PAMP.BA,0.20,0.120320,0.185045,run_20260421_141440_1b65423a,2026-04-21 14:14:40.010702+00:00
4,portfolio_run_20260421_141440_1b65423a,YPFD.BA,0.25,0.157178,0.225551,run_20260421_141440_1b65423a,2026-04-21 14:14:40.010702+00:00


# 10. Portfolio Type Classification

Simple rule-based categories are used for explainability in academic and business settings.

In [10]:
def classify_stock(row: pd.Series) -> str:
    if row["volatility"] < 0.25 and row["beta_vs_merval"] < 0.8:
        return "low-volatility"
    if row["sharpe_ratio"] >= 1.0 and row["volatility"] < 0.35:
        return "conservative"
    if row["beta_vs_merval"] > 1.2 and row["average_return"] > 0.20:
        return "growth"
    if row["volatility"] > 0.45:
        return "aggressive"
    return "balanced"


def classify_portfolio(expected_return: float, volatility: float, weighted_beta: float) -> str:
    if volatility < 0.20 and weighted_beta < 0.9:
        return "conservative"
    if expected_return > 0.25 and volatility > 0.40:
        return "aggressive"
    if expected_return > 0.20 and weighted_beta >= 1.1:
        return "growth"
    if volatility < 0.28:
        return "low-volatility"
    return "balanced"


stock_metrics["stock_type"] = stock_metrics.apply(classify_stock, axis=1)
portfolio_scenarios["portfolio_type"] = portfolio_scenarios.apply(
    lambda r: classify_portfolio(r["expected_portfolio_return"], r["portfolio_volatility"], r["weighted_beta"]),
    axis=1
)

stock_rankings = stock_metrics.sort_values(["sharpe_ratio", "average_return"], ascending=[False, False]).reset_index(drop=True)
stock_rankings["rank"] = stock_rankings.index + 1

display(stock_rankings)
display(portfolio_scenarios)

,ticker,average_return,volatility,beta_vs_merval,beta_vs_eem,corr_with_merval,corr_with_fx,sharpe_ratio,max_drawdown,sortino_ratio,calmar_ratio,inflation_proxy,country_risk_proxy,stock_type,rank
0,YPFD.BA,0.628713,0.570345,0.972664,0.890568,0.812950,0.015881,1.052357,-0.730358,1.520809,0.860829,0.002258,-0.000093,aggressive,1
1,PAMP.BA,0.601602,0.563061,1.028409,0.686464,0.870155,0.042220,1.017821,-0.520472,1.293475,1.155877,0.002258,-0.000093,aggressive,2
2,CEPU.BA,0.570019,0.626836,1.115562,0.806302,0.848996,0.039385,0.863881,-0.659236,1.100317,0.864666,0.002258,-0.000093,aggressive,3
3,GGAL.BA,0.541546,0.627942,1.192401,0.958328,0.904586,0.007596,0.817018,-0.684045,1.029080,0.791683,0.002258,-0.000093,aggressive,4
4,BMA.BA,0.533550,0.659160,1.226500,0.962794,0.886416,0.038960,0.766193,-0.614650,0.998121,0.868055,0.002258,-0.000093,growth,5


,portfolio_id,run_id,expected_portfolio_return,portfolio_volatility,weighted_beta,weighted_sharpe,diversification_effect,concentration_risk_hhi,num_assets,ingestion_timestamp,portfolio_type
0,portfolio_run_20260421_141440_1b65423a,run_20260421_141440_1b65423a,0.578421,0.536886,1.098257,1.024266,0.068198,0.21,5,2026-04-21 14:14:40.010702+00:00,aggressive


# 11. Upload Processed and Analytics Tables to BigQuery

In [11]:
# Determine write disposition based on INCREMENTAL_MODE
# False (default): WRITE_TRUNCATE for full reload
# True: WRITE_APPEND for incremental updates
processed_write_mode = "WRITE_APPEND" if INCREMENTAL_MODE else "WRITE_TRUNCATE"
analytics_write_mode = "WRITE_APPEND" if INCREMENTAL_MODE else "WRITE_TRUNCATE"

# Processed layer uploads.
to_bq(client, processed_asset_returns.dropna(subset=["date", "ticker", "log_return"]), PROJECT_ID, DATASETS["processed"], PROCESSED_TABLES["asset_returns"], processed_write_mode)
to_bq(client, processed_factor_returns.dropna(subset=["date", "factor_name", "factor_return"]), PROJECT_ID, DATASETS["processed"], PROCESSED_TABLES["factor_returns"], processed_write_mode)
to_bq(client, stock_metrics, PROJECT_ID, DATASETS["processed"], PROCESSED_TABLES["stock_metrics"], processed_write_mode)
to_bq(client, beta_metrics, PROJECT_ID, DATASETS["processed"], PROCESSED_TABLES["beta_metrics"], processed_write_mode)
to_bq(client, correlation_matrix_long, PROJECT_ID, DATASETS["processed"], PROCESSED_TABLES["correlation_matrix_long"], processed_write_mode)

# Analytics/serving layer uploads (always append for run history)
to_bq(client, portfolio_inputs, PROJECT_ID, DATASETS["analytics"], ANALYTICS_TABLES["portfolio_inputs"], "WRITE_APPEND")
to_bq(client, portfolio_scenarios, PROJECT_ID, DATASETS["analytics"], ANALYTICS_TABLES["portfolio_scenarios"], "WRITE_APPEND")
to_bq(client, stock_rankings, PROJECT_ID, DATASETS["analytics"], ANALYTICS_TABLES["stock_rankings"], "WRITE_APPEND")

print("Processed and analytics layers uploaded successfully.")

Uploaded 9,745 rows -> bigdata-financeargentina.processed_market.asset_returns (WRITE_TRUNCATE)
Uploaded 10,057 rows -> bigdata-financeargentina.processed_market.factor_returns (WRITE_TRUNCATE)
Uploaded 5 rows -> bigdata-financeargentina.processed_market.stock_metrics (WRITE_TRUNCATE)
Uploaded 5 rows -> bigdata-financeargentina.processed_market.beta_metrics (WRITE_TRUNCATE)
Uploaded 25 rows -> bigdata-financeargentina.processed_market.correlation_matrix_long (WRITE_TRUNCATE)
Uploaded 5 rows -> bigdata-financeargentina.analytics_market.portfolio_inputs (WRITE_APPEND)
Uploaded 1 rows -> bigdata-financeargentina.analytics_market.portfolio_scenarios (WRITE_APPEND)
Schema evolution enabled for bigdata-financeargentina.analytics_market.stock_rankings. Adding columns: sortino_ratio, calmar_ratio
Uploaded 5 rows -> bigdata-financeargentina.analytics_market.stock_rankings (WRITE_APPEND)
Processed and analytics layers uploaded successfully.


# 12. dbt Layer

This repository includes a dbt Core + dbt-bigquery scaffold with:
- dbt_project.yml
- profiles.yml.template
- models/sources.yml
- models/staging/stg_stock_prices.sql
- models/staging/stg_factor_prices.sql
- models/intermediate/int_asset_returns.sql
- models/marts/mart_stock_metrics.sql
- models/marts/mart_portfolio_scenarios.sql
- schema tests for not_null and unique constraints

Open those files in the project root to review and customize placeholders.

In [12]:
# Optional helper cell: print expected dbt artifacts to confirm scaffold.
dbt_files = [
    "dbt_project.yml",
    "profiles.yml.template",
    "models/sources.yml",
    "models/staging/stg_stock_prices.sql",
    "models/staging/stg_factor_prices.sql",
    "models/intermediate/int_asset_returns.sql",
    "models/marts/mart_stock_metrics.sql",
    "models/marts/mart_portfolio_scenarios.sql",
    "models/schema.yml"
]

for f in dbt_files:
    print(f)

dbt_project.yml
profiles.yml.template
models/sources.yml
models/staging/stg_stock_prices.sql
models/staging/stg_factor_prices.sql
models/intermediate/int_asset_returns.sql
models/marts/mart_stock_metrics.sql
models/marts/mart_portfolio_scenarios.sql
models/schema.yml


# 13. Example Terminal Commands

Run these in your terminal (not inside notebook cells):

```bash
dbt debug
dbt run
dbt test
dbt docs generate
```